# Data Cleaning and Feature Engineering


In [1]:
import os
project_root = os.path.abspath('..')
os.chdir(project_root)
print("Working directory:", os.getcwd())

Working directory: /Users/ac/Desktop/postharvest-iq


In [2]:
import os
import numpy as np
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()

engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

print("Connected to MySQL successfully")

Connected to MySQL successfully


In [3]:
# Load cereal prices — Northern markets + Southern reference markets
prices = pd.read_sql("""
    SELECT p.date, p.market, p.market_id, p.commodity,
           p.price, p.pricetype, p.currency,
           m.latitude, m.longitude, m.admin1
    FROM wfp_prices p
    JOIN wfp_markets m ON p.market_id = m.market_id
    WHERE p.commodity IN ('Maize', 'Millet', 'Sorghum')
    AND p.pricetype = 'Wholesale'
    AND p.market IN ('Tamale', 'Bolga', 'Wa', 'Kumasi', 'Techiman')
    ORDER BY p.market, p.commodity, p.date
""", engine)

prices['date'] = pd.to_datetime(prices['date'])
prices['year'] = prices['date'].dt.year
prices['month'] = prices['date'].dt.month

print(f"Total records: {len(prices)}")
print(f"Markets: {prices['market'].unique()}")
print(f"Commodities: {prices['commodity'].unique()}")
print(f"Date range: {prices['date'].min()} to {prices['date'].max()}")
print(f"Missing values: {prices.isnull().sum().sum()}")
print()
print("Records per market per commodity:")
print(prices.groupby(['market','commodity']).size().to_string())

Total records: 1770
Markets: ['Bolga' 'Kumasi' 'Tamale' 'Techiman' 'Wa']
Commodities: ['Maize' 'Millet' 'Sorghum']
Date range: 2006-01-15 00:00:00 to 2023-07-15 00:00:00
Missing values: 0

Records per market per commodity:
market    commodity
Bolga     Maize        108
          Millet        78
          Sorghum       71
Kumasi    Maize        206
          Millet       188
          Sorghum      127
Tamale    Maize        100
          Millet       129
          Sorghum       76
Techiman  Maize        163
          Millet       195
          Sorghum      118
Wa        Maize         67
          Millet        74
          Sorghum       70


In [4]:
# Load exchange rates — annual values only
fx = pd.read_sql("""
    SELECT year, value as exchange_rate
    FROM ghana_exchange_rates
    WHERE months = 'Annual value'
    ORDER BY year
""", engine)

fx['year'] = fx['year'].astype(int)

# Ghana redenominated the cedi in 2007 — exclude the transition year and earlier
# (2005-2007 rows contain both old and new denomination values; from 2008 only new)
fx_clean = fx[fx['year'] >= 2008].copy()
fx_clean = fx_clean.groupby('year')['exchange_rate'].mean().reset_index()

print(f"Exchange rate records: {len(fx_clean)}")
print(fx_clean)

# Merge exchange rates on year; fx_clean has one row per year after grouping
prices = prices.merge(fx_clean, on='year', how='left')

print(f"\nAfter merge: {len(prices)} rows")
print(f"Missing exchange rates: {prices['exchange_rate'].isnull().sum()}")

Exchange rate records: 18
    year  exchange_rate
0   2008       1.052275
1   2009       1.404967
2   2010       1.429983
3   2011       1.520625
4   2012       1.824867
5   2013       1.981350
6   2014       2.896575
7   2015       3.714642
8   2016       3.909817
9   2017       4.350533
10  2018       4.585325
11  2019       5.217367
12  2020       5.595708
13  2021       5.805700
14  2022       8.272400
15  2023      11.020409
16  2024      14.181942
17  2025      12.527120

After merge: 1770 rows
Missing exchange rates: 214


In [5]:
# Load producer prices for maize, millet, and sorghum
producer = pd.read_sql("""
    SELECT year, item, value as producer_price_index
    FROM fao_producer_prices
    WHERE item IN ('Maize (corn)', 'Millet', 'Sorghum')
    AND months = 'Annual value'
    ORDER BY item, year
""", engine)

producer['year'] = producer['year'].astype(int)
producer['item'] = producer['item'].replace('Maize (corn)', 'Maize')

# Ghana redenominated the cedi in 2007 — exclude the transition year and earlier
producer_clean = producer[producer['year'] >= 2008].copy()

# One value per year per item
producer_clean = producer_clean.groupby(
    ['year', 'item']
)['producer_price_index'].mean().reset_index()
producer_clean = producer_clean.rename(columns={'item': 'commodity'})

print(f"Producer price records: {len(producer_clean)}")
print(producer_clean.pivot(index='year', columns='commodity', values='producer_price_index'))

# Merge onto prices on year + commodity
prices = prices.merge(producer_clean, on=['year', 'commodity'], how='left')

print(f"\nAfter merge: {len(prices)} rows")
print(f"Missing producer index: {prices['producer_price_index'].isnull().sum()}")

Producer price records: 54
commodity      Maize     Millet    Sorghum
year                                      
2008        354.1225   489.3625   408.6575
2009        375.2925   552.1250   457.2650
2010        338.4125   559.8575   464.7325
2011        444.2850   608.1450   521.5650
2012         60.3400    61.7400    65.2200
2013         74.1700    76.3200    78.8100
2014         84.8500    86.0700    88.9900
2015        101.5500   102.7700   101.3000
2016        862.6500  1113.0400   877.1775
2017        639.5150   775.5075   569.8950
2018        745.8075   999.6475   826.9850
2019        597.6325   978.1950   757.3775
2020        925.1475  1152.9550   979.8625
2021       1357.1400  1837.7550  1483.3600
2022       2063.6075  2633.4500  2106.7750
2023        377.2500   364.9700   351.4600
2024        409.4000   384.7800   374.6000
2025        441.5600   475.6300   455.1700

After merge: 1770 rows
Missing producer index: 214


In [6]:
# Remove outliers per market per commodity using IQR method
n_before = len(prices)
print(f"Records before outlier removal: {n_before}")

def remove_outliers(group):
    Q1 = group['price'].quantile(0.25)
    Q3 = group['price'].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 3 * IQR
    upper = Q3 + 3 * IQR
    return group[
        (group['price'] >= lower) &
        (group['price'] <= upper)
    ]

prices = prices.groupby(
    ['market', 'commodity'], group_keys=False
).apply(remove_outliers).reset_index(drop=True)

print(f"Records after outlier removal: {len(prices)}")
print(f"Records removed: {n_before - len(prices)}")
print()
print("Records per market per commodity after cleaning:")
print(prices.groupby(['market','commodity']).size().to_string())

Records before outlier removal: 1770
Records after outlier removal: 1673
Records removed: 97

Records per market per commodity after cleaning:
market    commodity
Bolga     Maize        102
          Millet        74
          Sorghum       69
Kumasi    Maize        195
          Millet       181
          Sorghum      121
Tamale    Maize         98
          Millet       119
          Sorghum       70
Techiman  Maize        155
          Millet       182
          Sorghum      111
Wa        Maize         61
          Millet        68
          Sorghum       67


/var/folders/68/cg8hvl455rngc_wr5vs3wmsh0000gn/T/ipykernel_3350/1780281116.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply(remove_outliers).reset_index(drop=True)


In [7]:
# Add time features
prices['month_sin'] = np.sin(2 * np.pi * prices['month'] / 12)
prices['month_cos'] = np.cos(2 * np.pi * prices['month'] / 12)

print(f"Month sin range: {prices['month_sin'].min():.2f} to {prices['month_sin'].max():.2f}")
print(f"Month cos range: {prices['month_cos'].min():.2f} to {prices['month_cos'].max():.2f}")

Month sin range: -1.00 to 1.00
Month cos range: -1.00 to 1.00


In [8]:
# Save clean unscaled dataset — scaling happens in training notebooks after train/test split
final_columns = [
    'date', 'market', 'commodity', 'price',
    'latitude', 'longitude',
    'exchange_rate', 'producer_price_index',
    'month_sin', 'month_cos'
]

df_final = prices[final_columns].copy().dropna()

os.makedirs('data/processed', exist_ok=True)
df_final.to_csv('data/processed/cereals_merged_clean.csv', index=False)

print(f"Total rows: {len(df_final)}")
print(f"Columns: {list(df_final.columns)}")
print(f"Missing values: {df_final.isnull().sum().sum()}")
print(f"Markets: {df_final['market'].unique()}")
print(f"Commodities: {df_final['commodity'].unique()}")
print(f"Date range: {df_final['date'].min()} to {df_final['date'].max()}")
print(f"CSV file: {os.path.exists('data/processed/cereals_merged_clean.csv')}")

Total rows: 1459
Columns: ['date', 'market', 'commodity', 'price', 'latitude', 'longitude', 'exchange_rate', 'producer_price_index', 'month_sin', 'month_cos']
Missing values: 0
Markets: ['Bolga' 'Kumasi' 'Tamale' 'Techiman' 'Wa']
Commodities: ['Maize' 'Millet' 'Sorghum']
Date range: 2008-01-15 00:00:00 to 2022-12-15 00:00:00
CSV file: True
